<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Op%C3%B3%C5%BAnienia_Krak%C3%B3w.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analiza Opóźnień Komunikacji Miejskiej w Krakowie (GTFS-RT)

Niniejszy notatnik służy do analizy rzeczywistych opóźnień komunikacji miejskiej w Krakowie. Dane pobierane są w czasie rzeczywistym z usług [GTFS-RT ZTP Kraków](https://gtfs.ztp.krakow.pl/). 

Wykorzystujemy:
- **TripUpdates** (format `.pb` - Protobuf), aby pozyskać estymowane czasy przyjazdów i porównać je do planowanych.
- **Dane statyczne (GTFS)** do podpięcia lokalizacji geo (przystanków) oraz nazw linii.

Notatnik wygeneruje interaktywne mapy ulic ukazujące natężenie opóźnień, a także odpowiednie statystyki i wykresy.

In [ ]:
# Jeśli nie masz ich zainstalowanych, odkomentuj i uruchom poniższą linijkę:
!pip install gtfs-realtime-bindings protobuf requests pandas plotly folium scipy osmnx networkx matplotlib
import requests
from google.transit import gtfs_realtime_pb2
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import TimestampedGeoJson
import osmnx as ox
import networkx as nx
import matplotlib.colors as mcolors
import datetime
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def fetch_gtfs_rt_delays():
    print("Pobieranie aktualnych danych GTFS-RT (TripUpdates) dla Krakowa...")

    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb"
    }

    delays_data = []

    for v_type, url in urls.items():
        print(f" Pobieranie danych dla: {v_type}...")
        try:
            feed = gtfs_realtime_pb2.FeedMessage()
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            feed.ParseFromString(response.content)
            
            for entity in feed.entity:
                if entity.HasField('trip_update'):
                    trip_id = entity.trip_update.trip.trip_id
                    route_id = entity.trip_update.trip.route_id
                    
                    for stu in entity.trip_update.stop_time_update:
                        stop_id = stu.stop_id
                        delay = None
                        
                        # Pobieranie opóźnienia z przyjazdu lub odjazdu (preferujemy przyjazd)
                        if stu.HasField('arrival') and stu.arrival.HasField('delay'):
                            delay = stu.arrival.delay
                        elif stu.HasField('departure') and stu.departure.HasField('delay'):
                            delay = stu.departure.delay
                        
                        if delay is not None:
                            arr_time = None
                            if stu.HasField('arrival') and stu.arrival.HasField('time'):
                                arr_time = stu.arrival.time
                            elif stu.HasField('departure') and stu.departure.HasField('time'):
                                arr_time = stu.departure.time
                            
                            delays_data.append({
                                "typ": v_type,
                                "trip_id": trip_id,
                                "line_num": route_id,
                                "stop_sequence": stu.stop_sequence if stu.HasField('stop_sequence') else 0,
                                "stop_id": stop_id,
                                "delay_sec": delay,
                                "delay_min": delay / 60.0,
                                "time": arr_time
                            })
        except Exception as e:
            print(f"  Błąd podczas pobierania {v_type}: {e}")

    df_delays = pd.DataFrame(delays_data)
    
    if not df_delays.empty:
        # Posiadamy delay_sec (opóźnienia dodatnie oznaczają spóźnienie, pomijamy te < 0, bo to znaczy przyspieszenie)
        df_delays = df_delays[df_delays['delay_sec'] > 0]
        
        # Konwersja czasu Uniksowego do datetime i godziny ISO
        if 'time' in df_delays.columns:
            df_delays['datetime'] = pd.to_datetime(df_delays['time'], unit='s')
            df_delays['hour'] = df_delays['datetime'].dt.strftime('%Y-%m-%dT%H:00:00')
            # Jeżeli null (brak time) to przypisujemy bieżącą
            df_delays['hour'] = df_delays['hour'].fillna(datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00'))
        else:
            df_delays['hour'] = datetime.datetime.now().strftime('%Y-%m-%dT%H:00:00')
            
        print(f"\nZakończono. Pobrano {len(df_delays)} rekordów z dodatnimi opóźnieniami.")
    else:
        print("\nNie udało się pobrać żadnych opóźnień lub brak opóźnień w tej chwili.")
        
    return df_delays

df_delays = fetch_gtfs_rt_delays()
df_delays.head()

In [ ]:
# Wczytanie fizycznych lokalizacji przystanków i nazw linii z rozkładów (GTFS Zip)
# Zakładamy, że historyczne (ale w miarę aktualne) pliki przystanków znajdują się lokalnie
static_gtfs_dir = "data/GTFS_ZTP_17.05.26/"
stops_file = os.path.join(static_gtfs_dir, "stops.txt")
routes_file = os.path.join(static_gtfs_dir, "routes.txt")

if os.path.exists(stops_file) and not df_delays.empty:
    df_stops = pd.read_csv(stops_file, dtype=str)
    # Konwersja coords na wartości numeryczne
    df_stops["stop_lat"] = pd.to_numeric(df_stops["stop_lat"], errors="coerce")
    df_stops["stop_lon"] = pd.to_numeric(df_stops["stop_lon"], errors="coerce")
    
    # Łączenie przystanków
    df_merged = df_delays.merge(
        df_stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
        on='stop_id', 
        how='inner'
    )
    
    # Przypisywanie nazw linii, jeżeli istnieje routes.txt
    if os.path.exists(routes_file):
        df_routes = pd.read_csv(routes_file, dtype=str)
        df_merged = df_merged.merge(
            df_routes[['route_id', 'route_short_name']], 
            left_on='line_num', 
            right_on='route_id',
            how='left'
        )
        # Zastąpienie wewnętrznego ID linii jej nazwą publiczną np. "152"
        df_merged['linia'] = df_merged['route_short_name'].fillna(df_merged['line_num'])
    else:
        df_merged['linia'] = df_merged['line_num']

    # Obliczanie średniego opóźnienia, maksymalnego, oraz ilości pojazdów per Przystanek
    df_stops_delays = df_merged.groupby(['stop_name', 'stop_lat', 'stop_lon', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        max_delay_min=('delay_min', 'max'),
        measurements_count=('delay_min', 'count')
    )
    
    # Wyświetlamy tylko te przystanki, przez które opóźnione przejeżdża więcej niż x pojazdów
    df_stops_delays = df_stops_delays[df_stops_delays['measurements_count'] >= 2]
    
    display(df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head())
else:
    print("Brak pliku przystanków lub brak punktów pobranych - upewnij się, ze ścieżka do stops.txt jest poprawna.")

## Dynamiczna mapa opóźnień komunikacji (OSMnx Routes & Folium)
Poniżej generujemy interaktywną mapę animowaną w czasie. Wyszukujemy chronologiczne opóźnienia pojazdów w ramach konkretnych tras (na stykach przystanków) poprzez sieć dróg Krakowa!

In [ ]:
if 'df_merged' in locals() and not df_merged.empty:
    # 1. Sortujemy opóźnienia chronologicznie per trasa zeby móc narysować trasy per trip_id na konkretną godzinę
    df_trips = df_merged.sort_values(by=['trip_id', 'stop_sequence']).copy()
    
    # Do ucinania skomplikowanych obliczeń bierzemy tylko część topowych wycinków do pokazania (zbyt duży graf zatnie Colaba)
    # Wybieramy np. 20 tripów z największymi uśrednionymi opóźnieniami
    top_delayed_trips = df_trips.groupby('trip_id')['delay_min'].mean().nlargest(20).index
    df_trips = df_trips[df_trips['trip_id'].isin(top_delayed_trips)]
    
    # 2. Pobranie grafu drogowego miasta z OSMnx
    lokalizacja = "Kraków, Poland"
    print(f"Pobieranie geometrii dróg dla: {lokalizacja}... ")
    G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)
    
    # 3. Przypisywanie przystanków GTFS do najbliższych węzłów skrzyżowań (nodes) OSM
    unique_stops = df_trips.drop_duplicates(subset=['stop_id']).copy()
    try:
        nodes = ox.nearest_nodes(G, X=unique_stops['stop_lon'].values, Y=unique_stops['stop_lat'].values)
    except AttributeError:
        nodes = ox.distance.nearest_nodes(G, X=unique_stops['stop_lon'].values, Y=unique_stops['stop_lat'].values)
        
    stop_to_node = dict(zip(unique_stops['stop_id'], nodes))
    df_trips['osmid'] = df_trips['stop_id'].map(stop_to_node)
    
    print("Wyliczanie tras między przystankami (routing najkrótszych ścieżek)...")
    features = []
    cmap = plt.get_cmap('autumn_r') 
    norm = mcolors.Normalize(vmin=0, vmax=15) # Skala od 0 do 15 min opoznienia 
    
    grouped_by_trip = df_trips.groupby('trip_id')
    for trip_id, trip_data in grouped_by_trip:
        trip_data = trip_data.sort_values('stop_sequence')
        
        # Iteracja po parze przystanków (A -> B)
        nodes_list = trip_data['osmid'].tolist()
        delays_list = trip_data['delay_min'].tolist()
        hours_list = trip_data['hour'].tolist()
        
        for i in range(len(nodes_list) - 1):
            source = nodes_list[i]
            target = nodes_list[i+1]
            if source == target:
                continue
                
            try:
                # Wyszukujemy drogę po fizycznej siatce miasta między przystankiem początkowym a końcowym
                path = nx.shortest_path(G, source, target, weight='length')
                
                # Uśredniamy opóźnienie między obu przystankami
                avg_delay = (delays_list[i] + delays_list[i+1]) / 2.0
                hour_val = hours_list[i]
                
                coords = [[G.nodes[n]['x'], G.nodes[n]['y']] for n in path]
                
                features.append({
                    "type": "Feature",
                    "geometry": {
                        "type": "LineString",
                        "coordinates": coords
                    },
                    "properties": {
                        "times": [hour_val] * len(coords), 
                        "style": {
                            "color": mcolors.to_hex(cmap(norm(avg_delay))),
                            "weight": 5,
                            "opacity": 0.8
                        }
                    }
                })
            except nx.NetworkXNoPath:
                continue

    # 4. Generowanie animowanej mapy na tło CartoDB (bądź OpenStreetMap)
    print("Budowanie widoku The Animated Heatmap Routing...")
    folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=13, tiles="cartodbdark_matter")
    
    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="PT1H",
        add_last_point=False,
        auto_play=True,
        loop=True,
        max_speed=1,
        loop_button=True,
        time_slider_drag_update=True
    ).add_to(folium_map)
    
    display(folium_map)
else:
    print("Zbyt mało danych do wykonania animacji lub brak przystanków.")

## Wykresy (Gdzie są największe opóźnienia)
Przeanalizujmy, które przystanki oraz które linie notują średnio największe opóźnienia w pozyskanej próbce czasowej.

In [ ]:
if 'df_stops_delays' in locals() and not df_stops_delays.empty:
    # Top 15 Przystanków
    top_15_mean = df_stops_delays.sort_values(by='mean_delay_min', ascending=False).head(15)

    fig_bar_stops = px.bar(
        top_15_mean,
        x='stop_name',
        y='mean_delay_min',
        color='typ',
        title="Top 15 przystanków o największym średnim opóźnieniu",
        labels={'stop_name': 'Przystanek', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_stops.update_layout(xaxis_tickangle=-45)
    fig_bar_stops.show()
    
if 'df_merged' in locals() and not df_merged.empty:
    # Agregacja po Liniach
    df_route_delays = df_merged.groupby(['linia', 'typ'], as_index=False).agg(
        mean_delay_min=('delay_min', 'mean'),
        measurements_count=('delay_min', 'count')
    )
    
    # Filtrujemy by odrzucić pojedyncze strzały pomiarów dla jakiejś trasy
    df_route_delays = df_route_delays[df_route_delays['measurements_count'] >= 3]
    
    top_15_routes = df_route_delays.sort_values(by='mean_delay_min', ascending=False).head(15)
    
    fig_bar_routes = px.bar(
        top_15_routes,
        x='linia',
        y='mean_delay_min',
        color='typ',
        title="Top 15 linii komunikacyjnych o największym średnim opóźnieniu",
        labels={'linia': 'Numer Linii', 'mean_delay_min': 'Średnie opóźnienie (min)'},
        text_auto=':.1f',
        height=500
    )
    fig_bar_routes.update_layout(xaxis_type='category') # By numery linii zachowywały się jak kategorie
    fig_bar_routes.show()